In [1]:
import os
import gc
import csv
import glob
import h5py
import joblib
import random

In [2]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [3]:
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [4]:
from sklearn import metrics
from sklearn.metrics import auc
from sklearn.metrics import f1_score
from sklearn.metrics import roc_curve
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay 
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

In [5]:
import keras
import tensorflow as tf
import tensorflow.keras.backend as K

from keras.models import Model

from keras.optimizers import SGD
from keras.optimizers import Adam
from keras.optimizers import Nadam

from keras.optimizers.schedules import ExponentialDecay
from keras.optimizers.schedules import CosineDecay

from tensorflow.keras.regularizers import l2
from tensorflow.keras.regularizers import l1
from tensorflow.keras.regularizers import l1_l2 

from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.losses import BinaryFocalCrossentropy

from tensorflow.keras.models import Sequential
from tensorflow.keras.models import load_model

from tensorflow.keras.utils import to_categorical

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import ReduceLROnPlateau 

from tensorflow.keras.metrics import AUC
from tensorflow.keras.metrics import Recall
from tensorflow.keras.metrics import Precision 
from tensorflow.keras.metrics import TruePositives
from tensorflow.keras.metrics import TrueNegatives

from tensorflow.keras.layers import GRU
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Lambda
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import LeakyReLU
from tensorflow.keras.layers import Attention
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import InputLayer
from tensorflow.keras.layers import Concatenate
from tensorflow.keras.layers import GaussianNoise
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import GlobalAveragePooling1D

In [6]:
from typing import Any
from json import loads, dumps
from datetime import datetime, date

In [7]:
from tqdm.notebook import tqdm
from multiprocessing import Lock

tqdm.set_lock(Lock())

In [8]:
print("Is GPU available:", tf.config.list_physical_devices('GPU'))

Is GPU available: []


In [ ]:
class H5Sequence(tf.keras.utils.Sequence):

    def __init__(self, file_path, batch_size=32, **kwargs):
        super().__init__(**kwargs)

        self.file_path = file_path
        self.batch_size = batch_size
        self.file = h5py.File(self.file_path, "r")

        self.X = self.file["X"]
        self.y = self.file["y"]
        self.length = len(self.y)

    def __len__(self):
        return int(np.ceil(self.length / self.batch_size))

    def __getitem__(self, idx):
        start = idx * self.batch_size
        end = start + self.batch_size

        X = self.X[start:end].astype("float32")
        y = self.y[start:end]

        return X, y

In [9]:
file_ext = '.h5'
base_path = os.getcwd()

data_path = os.path.join(base_path, 'datasets_ready')
curr_path = os.path.join(base_path, 'data_preprocessing')

data_train_path = os.path.join(base_path, 'datasets_training')
data_test_path  = os.path.join(base_path, 'datasets_testing')

In [10]:
data_train = sorted([os.path.join(data_train_path, path) for path in os.listdir(data_train_path) if file_ext in path])
data_train

['E:\\Jupyter\\manannan\\stuck-prediction-v2\\datasets_training\\1. KMJ-69.4_train_6060.h5']

In [11]:
data_test = sorted([os.path.join(data_test_path, path) for path in os.listdir(data_test_path) if file_ext in path])
data_test

['E:\\Jupyter\\manannan\\stuck-prediction-v2\\datasets_testing\\1. KMJ-69.4_test_6060.h5']

In [12]:
batch_size = 512

train_dataset = H5Sequence(data_train[0], batch_size=batch_size, filtered=False)
test_dataset  = H5Sequence(data_test[0], batch_size=batch_size, filtered=False)

In [13]:
seed_value = 42

np.random.seed(seed_value)
random.seed(seed_value)
tf.random.set_seed(seed_value)

In [14]:
step_in    = 60
step_out   = 60
n_features = 89

In [15]:
#-- initialize hyperparameters
hyperparameters = {
    "model"                 : "gru",
    "epoch"                 : 10,
    "batch"                 : 128,
    "neurons"               : 8,
    "step_in"               : step_in,
    "step_out"              : step_out,
    "optimizer"             : "Adam",
    "constant_l"            : 0.0,
    "class_weight"          : False, 
    "loss_function"         : "BCE",
    "regularization"        : "None",
    "learning_rate"         : 0.001,
    "dropout_rate"          : 0.0,
    "dropout_rate_layer"    : 0.0,
    "dropout_rate_recurrent": 0.0,
}

In [16]:
#-- others hyperparameters
es_params = {
    "monitor"              : "val_recall",
    "patience"             : 100,
    "restore_best_weights" : True
}

In [17]:
def ExtractContext(x):
    return x[:, -1:, :]

def ExtractContextShape(input_shape):
    return (input_shape[0], 1, input_shape[2])

def GetContext(x):
    return tf.squeeze(x, axis=1)

def GetContextShape(input_shape):
    return (input_shape[0], input_shape[2])

In [18]:
if hyperparameters["model"] == 'gru':
    inputs = Input(shape=(step_in, n_features))
    gru    = GRU(hyperparameters["neurons"], return_sequences=False)(inputs)
    output = Dense(1, activation='sigmoid')(gru)
    model  = Model(inputs=inputs, outputs=output)

In [19]:
#-- add callbacks
early_stopping = EarlyStopping(**es_params)

In [20]:
#-- model params
model_params = {
    "loss"      : BinaryCrossentropy(),
    "optimizer" : Adam(learning_rate=hyperparameters["learning_rate"], clipnorm=1.0),
    "metrics"   : [
        "accuracy", 
        AUC(name = "auc"), 
        Precision(name = "precision"), 
        Recall(name = "recall")
    ]
}

In [21]:
#-- add class weight
if (hyperparameters["class_weight"]):
    class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train.flatten())
    class_weights = {0: 2, 1: 1}

    print(class_weights)

In [22]:
#-- compile model
model.compile(**model_params)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)             │ (None, 60, 89)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ gru (GRU)                            │ (None, 8)                   │           2,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │               9 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,385 (9.32 KB)

 Trainable params: 2,385 (9.32 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
#-- training params
model_params = {
    "epochs": hyperparameters["epoch"],
    "callbacks": [early_stopping],
}

if hyperparameters["class_weight"]:
    model_params["class_weight"] = class_weights

In [24]:
#-- train models
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    **model_params
)

E:\Jupyter\venv\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
940/940 ━━━━━━━━━━━━━━━━━━━━ 327s 345ms/step - accuracy: 0.9903 - auc: 0.5009 - loss: 0.0686 - precision: 0.0000e+00 - recall: 0.0000e+00 - val_accuracy: 0.9608 - val_auc: 0.4948 - val_loss: 0.1968 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00
Epoch 2/10
 39/940 ━━━━━━━━━━━━━━━━━━━━ 4:57 330ms/step - accuracy: 0.9321 - auc: 0.4815 - loss: 0.3349 - precision: 0.0000e+00 - recall: 0.0000e+00

KeyboardInterrupt: 